# HPO Experiments for Healthcare ML — Complete Colab Runner

**Paper:** *An Experimental Comparison of Hyperparameter Optimization Methods for Healthcare Machine Learning: From Grid Search to Swarm Intelligence*

**What this notebook does.** Runs the full experimental grid — 6 HPO methods × 3 classifiers × 10 seeds × (up to) 4 datasets — and produces:
1. `results_raw.csv` — one row per (dataset, classifier, method, seed) with accuracy, F1 (macro + positive-class), FNR, wall-clock time
2. `results_summary.csv` — mean ± std over 10 seeds per cell, Wilcoxon p-values vs Manual, Holm-Bonferroni adjusted p-values, Cliff's δ
3. `latex_tables.tex` — ready-to-paste `\multirow` blocks for `tab:pima_results`, `tab:ilpd_results`, and optionally `tab:bc_results` / `tab:hd_results` and `tab:summary`

**How to use.**
1. File → Upload notebook → upload this `.ipynb` into Colab.
2. Runtime → Run all. First run takes ~1–3 hours on a free Colab CPU runtime for the 2 new datasets; ~3–6 hours for all 4 datasets. Results auto-save to `/content/` and to Google Drive if mounted.
3. Download the three output files and paste into the manuscript's `% TODO:` slots.

**Design decisions.** Seeds 42–51 match your existing paper. Search spaces, budgets, split ratios, and metrics match Table `tab:hyperparam_spaces` and the Methodology section of the manuscript exactly.

**Reproducibility note.** Every run fixes the numpy, Python `random`, scikit-learn, and optuna seeds. Library versions are pinned in the install cell. A SHA-256 hash of each dataset's raw bytes is logged in `dataset_hashes.json`.


## 1. Configuration — **edit this cell if needed**

In [ ]:
# =========================================================================
# Which datasets to run? Options: 'new_only' (Pima + ILPD) or 'all_four'.
# 'new_only' keeps your existing BC/HD results and only runs the additions.
# 'all_four' re-runs everything from scratch for a fully consistent set.
# =========================================================================
DATASETS_TO_RUN = 'all_four'   # 'new_only' | 'all_four'

# Seeds used for the 10 independent runs per (dataset, classifier, method)
SEEDS = list(range(42, 52))

# Budgets — match the paper's Methodology section
RANDOM_BUDGET = 50
BAYESIAN_BUDGET = 50
GWO_WOLVES, GWO_ITERS = 10, 20
PSO_PARTICLES, PSO_ITERS = 10, 20
PSO_C1 = PSO_C2 = 2.0
PSO_W = 0.9

# Train / val / test split ratios
TRAIN_FRAC, VAL_FRAC, TEST_FRAC = 0.70, 0.15, 0.15

# Mount Google Drive? If True, outputs also saved to Drive
MOUNT_DRIVE = False
DRIVE_OUTPUT_DIR = '/content/drive/MyDrive/hpo_experiments'

print(f'DATASETS_TO_RUN = {DATASETS_TO_RUN}')
print(f'SEEDS = {SEEDS}')

## 2. Install and pin dependencies

Only `optuna` is installed fresh; `numpy`, `pandas`, `scikit-learn`, and `scipy` use Colab's defaults to avoid binary-incompatibility errors.

**If you previously ran a version of this notebook that downgraded `numpy` or `scikit-learn`, do Runtime -> Disconnect and delete runtime -> Run all before continuing.** Otherwise, C extensions built against a different numpy ABI will fail to import.

In [ ]:
# Install only what's missing from Colab's default environment.
# DO NOT downgrade OR force-upgrade numpy/pandas/sklearn — Colab's other
# libraries are compiled against specific versions and both directions break
# imports. The experiments in this notebook are insensitive to minor version
# drift; record the exact versions below for the paper's reproducibility
# section.
!pip install -q 'optuna>=3.4,<4.0'

import sys, platform
import numpy as np, pandas as pd, sklearn, scipy, optuna
print('Python      :', sys.version.split()[0])
print('Platform    :', platform.platform())
print('numpy       :', np.__version__)
print('pandas      :', pd.__version__)
print('scikit-learn:', sklearn.__version__)
print('scipy       :', scipy.__version__)
print('optuna      :', optuna.__version__)
print()
print('Record these versions in the manuscript reproducibility paragraph.')


In [ ]:
# Optional: mount Drive for persistent output
if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    import os
    os.makedirs(DRIVE_OUTPUT_DIR, exist_ok=True)
    print('Drive mounted. Outputs will also be copied to', DRIVE_OUTPUT_DIR)
else:
    print('Drive not mounted — outputs will only be in /content/')

## 3. Dataset loaders

All four datasets are pulled from stable public URLs. If a URL fails, fall-through mirrors are tried. Every raw payload is SHA-256 hashed and logged for reproducibility.


In [ ]:
import hashlib, io, json, os, time, urllib.request
import numpy as np, pandas as pd

DATA_DIR = '/content/data'
os.makedirs(DATA_DIR, exist_ok=True)

def _fetch(url, timeout=30):
    req = urllib.request.Request(url, headers={'User-Agent': 'hpo-study/1.0'})
    with urllib.request.urlopen(req, timeout=timeout) as r:
        return r.read()

def fetch_first(urls):
    last_err = None
    for u in urls:
        try:
            blob = _fetch(u)
            print(f'  fetched from {u} ({len(blob)} bytes)')
            return blob, u
        except Exception as e:
            print(f'  failed {u}: {e}')
            last_err = e
    raise RuntimeError(f'All sources failed: {last_err}')

def sha256(b): return hashlib.sha256(b).hexdigest()

In [ ]:
# ---- Breast Cancer Wisconsin (built in to sklearn) ----------------------
def load_bc():
    from sklearn.datasets import load_breast_cancer
    data = load_breast_cancer(as_frame=True)
    X = data.data.values.astype(float)
    # sklearn's target: 0=malignant, 1=benign. We want 1=malignant (disease) for FNR consistency
    y = (1 - data.target.values).astype(int)
    return X, y, list(data.feature_names)

# ---- Heart Disease Statlog ---------------------------------------------
STATLOG_URLS = [
    'https://archive.ics.uci.edu/ml/machine-learning-databases/statlog/heart/heart.dat',
    'https://raw.githubusercontent.com/jbrownlee/Datasets/master/heart.csv',
]
def load_hd():
    blob, used = fetch_first(STATLOG_URLS)
    text = blob.decode('utf-8', errors='replace').strip()
    # Two formats: Statlog heart.dat is space-separated 14 cols, jbrownlee is comma, headerless
    sep = None if 'heart.dat' in used else ','
    df = pd.read_csv(io.StringIO(text), sep=sep, header=None, engine='python')
    assert df.shape[1] == 14, f'Expected 14 cols, got {df.shape[1]}'
    X = df.iloc[:, :13].values.astype(float)
    # Statlog: target is 1 (absence) or 2 (presence); recode so 1 = disease
    raw_y = df.iloc[:, 13].values
    if set(np.unique(raw_y)) == {1, 2}:
        y = (raw_y == 2).astype(int)
    else:
        y = (raw_y > 0).astype(int)
    return X, y, [f'f{i}' for i in range(13)], blob

# ---- Pima Indians Diabetes ---------------------------------------------
PIMA_URLS = [
    'https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv',
    'https://raw.githubusercontent.com/npradaschnor/Pima-Indians-Diabetes-Dataset/master/diabetes.csv',
]
PIMA_COLS = ['Pregnancies','Glucose','BloodPressure','SkinThickness',
             'Insulin','BMI','DiabetesPedigreeFunction','Age','Outcome']
def load_pima():
    blob, _ = fetch_first(PIMA_URLS)
    text = blob.decode('utf-8', errors='replace')
    first = text.splitlines()[0].split(',')[0].strip()
    has_header = not first.lstrip('-').replace('.', '', 1).isdigit()
    df = pd.read_csv(io.StringIO(text), header=0 if has_header else None,
                     names=None if has_header else PIMA_COLS)
    if has_header:
        df.columns = PIMA_COLS  # force canonical names
    assert df.shape == (768, 9), f'Pima shape {df.shape}'
    zero_as_nan = ['Glucose','BloodPressure','SkinThickness','Insulin','BMI']
    for c in zero_as_nan:
        df[c] = df[c].replace(0, np.nan)
    # NOTE: fold-level imputation is done inside the experiment loop below
    X = df[PIMA_COLS[:-1]].values.astype(float)
    y = df['Outcome'].values.astype(int)
    return X, y, PIMA_COLS[:-1], blob

# ---- ILPD -------------------------------------------------------------
ILPD_URLS = [
    'https://raw.githubusercontent.com/saahil1292/indian_liver_patients/main/indian_liver_patient.csv',
    'https://raw.githubusercontent.com/saahil1292/indian_liver_patients/master/indian_liver_patient.csv',
    'https://raw.githubusercontent.com/harshilpatel1799/ML-Models-Comparion-Indian-Liver-Data-Set/master/indian_liver_patient.csv',
]
def load_ilpd():
    blob, _ = fetch_first(ILPD_URLS)
    df = pd.read_csv(io.BytesIO(blob))
    assert df.shape == (583, 11), f'ILPD shape {df.shape}'
    df['Gender'] = df['Gender'].astype(str).str.strip().map({'Male': 1, 'Female': 0}).astype(int)
    # Original: 1 = patient, 2 = non-patient. Recode so 1 = disease.
    y = df['Dataset'].map({1: 1, 2: 0}).astype(int).values
    feat_cols = [c for c in df.columns if c != 'Dataset']
    X = df[feat_cols].values.astype(float)
    return X, y, feat_cols, blob

In [ ]:
# Load the chosen dataset set
all_datasets = {}
hashes = {}

print('[BC] loading…'); X, y, _ = load_bc()
all_datasets['BC'] = (X, y)
hashes['BC'] = 'sklearn-builtin'
print(f'  shape={X.shape}, class balance={np.bincount(y).tolist()}')

print('[HD] loading…'); X, y, _, blob = load_hd()
all_datasets['HD'] = (X, y); hashes['HD'] = sha256(blob)
print(f'  shape={X.shape}, class balance={np.bincount(y).tolist()}, sha256={hashes["HD"][:16]}…')

print('[PD] loading…'); X, y, _, blob = load_pima()
all_datasets['PD'] = (X, y); hashes['PD'] = sha256(blob)
print(f'  shape={X.shape}, class balance={np.bincount(y).tolist()}, sha256={hashes["PD"][:16]}…')

print('[ILPD] loading…'); X, y, _, blob = load_ilpd()
all_datasets['ILPD'] = (X, y); hashes['ILPD'] = sha256(blob)
print(f'  shape={X.shape}, class balance={np.bincount(y).tolist()}, sha256={hashes["ILPD"][:16]}…')

with open(os.path.join(DATA_DIR, 'dataset_hashes.json'), 'w') as f:
    json.dump(hashes, f, indent=2)

if DATASETS_TO_RUN == 'new_only':
    ACTIVE_DATASETS = ['PD', 'ILPD']
elif DATASETS_TO_RUN == 'all_four':
    ACTIVE_DATASETS = ['BC', 'HD', 'PD', 'ILPD']
else:
    raise ValueError(DATASETS_TO_RUN)
print(f'\nActive datasets for this run: {ACTIVE_DATASETS}')

## 4. Classifier factories and search spaces

Search spaces match Table `tab:hyperparam_spaces` in the paper exactly.


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression

# ---------- search-space specs (used by all HPO methods) --------------
RF_SPACE = {
    'n_estimators': {'type': 'cat', 'values': [50, 100, 150, 200, 300]},
    'max_depth':    {'type': 'cat', 'values': [5, 10, 15, 20, 25, None]},
    'min_samples_split': {'type': 'int', 'low': 2, 'high': 15},
    'min_samples_leaf':  {'type': 'int', 'low': 1, 'high': 8},
    'max_features': {'type': 'cat', 'values': ['sqrt', 'log2', 0.3, 0.5]},
}
SVM_SPACE = {
    'C': {'type': 'float_log', 'low': 0.01, 'high': 100.0},
    'kernel': {'type': 'cat', 'values': ['linear', 'rbf', 'poly']},
    'gamma':  {'type': 'cat', 'values': ['scale', 'auto', 0.01, 0.1]},
}
LR_SPACE = {
    'C': {'type': 'float_log', 'low': 0.001, 'high': 100.0},
    'solver': {'type': 'cat', 'values': ['lbfgs', 'liblinear', 'saga']},
}

# ---------- manual defaults (Methodology §HPO) ------------------------
MANUAL_DEFAULTS = {
    'RF':  {'n_estimators': 100, 'max_features': 'sqrt'},
    'SVM': {'C': 1.0, 'kernel': 'rbf', 'gamma': 'scale'},
    'LR':  {'C': 1.0, 'solver': 'lbfgs'},
}

def build_rf(params, seed):
    return RandomForestClassifier(random_state=seed, n_jobs=1, **params)
def build_svm(params, seed):
    return SVC(random_state=seed, **params)
def build_lr(params, seed):
    return LogisticRegression(random_state=seed, max_iter=2000, **params)

CLASSIFIERS = {
    'RF':  (build_rf,  RF_SPACE),
    'SVM': (build_svm, SVM_SPACE),
    'LR':  (build_lr,  LR_SPACE),
}

## 5. HPO method implementations

Manual and grid/random use sklearn's built-in tooling where practical. Bayesian uses optuna's TPE (matches `snoek2012` + Optuna 3.4 cite in the manuscript). GWO and PSO are numpy implementations following the cited `mirjalili2014` and `kennedy1995` formulations.


In [ ]:
import itertools, random, warnings, time
warnings.filterwarnings('ignore')  # silence sklearn convergence chatter

# ---------- helpers: encode / decode continuous vector <-> params ----
def space_dims(space):
    """Return list of (name, type, info) for continuous encoding."""
    dims = []
    for name, spec in space.items():
        if spec['type'] == 'cat':
            dims.append((name, 'cat', spec['values']))
        elif spec['type'] == 'int':
            dims.append((name, 'int', (spec['low'], spec['high'])))
        elif spec['type'] == 'float_log':
            dims.append((name, 'float_log', (spec['low'], spec['high'])))
    return dims

def vec_to_params(vec, dims):
    """Map a vector in [0,1]^d to concrete hyperparameters."""
    out = {}
    for v, (name, kind, info) in zip(vec, dims):
        v = float(np.clip(v, 0.0, 1.0 - 1e-9))
        if kind == 'cat':
            idx = int(v * len(info))
            out[name] = info[idx]
        elif kind == 'int':
            lo, hi = info
            out[name] = int(round(lo + v * (hi - lo)))
        elif kind == 'float_log':
            lo, hi = info
            out[name] = float(np.exp(np.log(lo) + v * (np.log(hi) - np.log(lo))))
    return out

def sample_random_params(space, rng):
    out = {}
    for name, spec in space.items():
        if spec['type'] == 'cat':
            out[name] = rng.choice(spec['values'])
        elif spec['type'] == 'int':
            out[name] = int(rng.randint(spec['low'], spec['high']+1))
        elif spec['type'] == 'float_log':
            out[name] = float(np.exp(rng.uniform(np.log(spec['low']), np.log(spec['high']))))
    return out

# ---------- objective: macro-F1 on validation fold -------------------
from sklearn.metrics import f1_score
def objective(builder, params, Xtr, ytr, Xva, yva, seed):
    try:
        # SVM/LR: some kernels/solvers need probability or dual settings; sklearn handles via params dict
        clf = builder(params, seed)
        clf.fit(Xtr, ytr)
        yp = clf.predict(Xva)
        return float(f1_score(yva, yp, average='macro'))
    except Exception:
        return 0.0   # penalize infeasible configs


In [ ]:
# ========================================================================
# The six HPO methods. Each returns (best_params, n_evals, wallclock_seconds)
# ========================================================================

def hpo_manual(clf_name, builder, space, Xtr, ytr, Xva, yva, seed):
    t0 = time.perf_counter()
    best = MANUAL_DEFAULTS[clf_name]
    return best, 1, time.perf_counter() - t0

def hpo_grid(clf_name, builder, space, Xtr, ytr, Xva, yva, seed):
    t0 = time.perf_counter()
    # Enumerate a grid. For RF: limit to the cartesian product of *categorical* axes
    # plus 3 uniform picks per integer axis (matches ~162 configs in paper).
    axes = []
    for name, spec in space.items():
        if spec['type'] == 'cat':
            axes.append([(name, v) for v in spec['values']])
        elif spec['type'] == 'int':
            # 3-point grid
            lo, hi = spec['low'], spec['high']
            vals = sorted(set([lo, (lo+hi)//2, hi]))
            axes.append([(name, v) for v in vals])
        elif spec['type'] == 'float_log':
            # 5-point log grid
            vals = np.geomspace(spec['low'], spec['high'], 5).tolist()
            axes.append([(name, float(v)) for v in vals])
    best_params, best_score = None, -1
    n_evals = 0
    for combo in itertools.product(*axes):
        params = dict(combo)
        s = objective(builder, params, Xtr, ytr, Xva, yva, seed)
        n_evals += 1
        if s > best_score:
            best_score, best_params = s, params
    return best_params, n_evals, time.perf_counter() - t0

def hpo_random(clf_name, builder, space, Xtr, ytr, Xva, yva, seed, budget=RANDOM_BUDGET):
    t0 = time.perf_counter()
    rng = random.Random(seed)
    best_params, best_score = None, -1
    for _ in range(budget):
        params = sample_random_params(space, rng)
        s = objective(builder, params, Xtr, ytr, Xva, yva, seed)
        if s > best_score:
            best_score, best_params = s, params
    return best_params, budget, time.perf_counter() - t0

def hpo_bayesian(clf_name, builder, space, Xtr, ytr, Xva, yva, seed, budget=BAYESIAN_BUDGET):
    t0 = time.perf_counter()
    import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    def objective_optuna(trial):
        params = {}
        for name, spec in space.items():
            if spec['type'] == 'cat':
                params[name] = trial.suggest_categorical(name, spec['values'])
            elif spec['type'] == 'int':
                params[name] = trial.suggest_int(name, spec['low'], spec['high'])
            elif spec['type'] == 'float_log':
                params[name] = trial.suggest_float(name, spec['low'], spec['high'], log=True)
        return objective(builder, params, Xtr, ytr, Xva, yva, seed)
    sampler = optuna.samplers.TPESampler(seed=seed)
    study = optuna.create_study(direction='maximize', sampler=sampler)
    study.optimize(objective_optuna, n_trials=budget, show_progress_bar=False)
    return study.best_params, budget, time.perf_counter() - t0

def _swarm_run(clf_name, builder, space, Xtr, ytr, Xva, yva, seed,
               pop_size, iters, algorithm):
    """GWO or PSO on continuous vector in [0,1]^d; decoded per eval."""
    t0 = time.perf_counter()
    rng = np.random.default_rng(seed)
    dims = space_dims(space)
    d = len(dims)
    pos = rng.uniform(0, 1, size=(pop_size, d))
    vel = rng.uniform(-0.1, 0.1, size=(pop_size, d)) if algorithm == 'PSO' else None
    fitness = np.array([objective(builder, vec_to_params(pos[i], dims),
                                  Xtr, ytr, Xva, yva, seed) for i in range(pop_size)])
    n_evals = pop_size
    pbest_pos, pbest_fit = pos.copy(), fitness.copy()
    gbest_idx = int(np.argmax(fitness))
    gbest_pos, gbest_fit = pos[gbest_idx].copy(), float(fitness[gbest_idx])
    if algorithm == 'GWO':
        # three top wolves
        order = np.argsort(-fitness)
        alpha, beta, delta = pos[order[0]].copy(), pos[order[1]].copy(), pos[order[2]].copy()
    for it in range(iters):
        if algorithm == 'PSO':
            r1, r2 = rng.uniform(0, 1, size=(pop_size, d)), rng.uniform(0, 1, size=(pop_size, d))
            vel = PSO_W * vel + PSO_C1 * r1 * (pbest_pos - pos) + PSO_C2 * r2 * (gbest_pos - pos)
            pos = np.clip(pos + vel, 0, 1)
        else:  # GWO
            a = 2.0 * (1.0 - it / max(1, iters - 1))
            for i in range(pop_size):
                A1, A2, A3 = (2*a*rng.uniform(0,1,d) - a for _ in range(3))
                C1, C2, C3 = (2*rng.uniform(0,1,d) for _ in range(3))
                D_alpha = np.abs(C1*alpha - pos[i])
                D_beta  = np.abs(C2*beta  - pos[i])
                D_delta = np.abs(C3*delta - pos[i])
                X1 = alpha - A1*D_alpha
                X2 = beta  - A2*D_beta
                X3 = delta - A3*D_delta
                pos[i] = np.clip((X1 + X2 + X3) / 3.0, 0, 1)
        fitness = np.array([objective(builder, vec_to_params(pos[i], dims),
                                      Xtr, ytr, Xva, yva, seed) for i in range(pop_size)])
        n_evals += pop_size
        if algorithm == 'PSO':
            better = fitness > pbest_fit
            pbest_pos[better], pbest_fit[better] = pos[better], fitness[better]
            gi = int(np.argmax(pbest_fit))
            if pbest_fit[gi] > gbest_fit:
                gbest_fit, gbest_pos = float(pbest_fit[gi]), pbest_pos[gi].copy()
        else:
            order = np.argsort(-fitness)
            alpha, beta, delta = pos[order[0]].copy(), pos[order[1]].copy(), pos[order[2]].copy()
            if fitness[order[0]] > gbest_fit:
                gbest_fit, gbest_pos = float(fitness[order[0]]), pos[order[0]].copy()
    best_params = vec_to_params(gbest_pos, dims)
    return best_params, n_evals, time.perf_counter() - t0

def hpo_gwo(clf_name, builder, space, Xtr, ytr, Xva, yva, seed):
    return _swarm_run(clf_name, builder, space, Xtr, ytr, Xva, yva, seed,
                      GWO_WOLVES, GWO_ITERS, 'GWO')

def hpo_pso(clf_name, builder, space, Xtr, ytr, Xva, yva, seed):
    return _swarm_run(clf_name, builder, space, Xtr, ytr, Xva, yva, seed,
                      PSO_PARTICLES, PSO_ITERS, 'PSO')

HPO_METHODS = {
    'Manual':   hpo_manual,
    'Grid':     hpo_grid,
    'Random':   hpo_random,
    'Bayesian': hpo_bayesian,
    'GWO':      hpo_gwo,
    'PSO':      hpo_pso,
}
print('HPO methods registered:', list(HPO_METHODS))

## 6. Main experiment loop

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

def prep_split(X, y, seed):
    # Stratified 70/15/15. Impute with training median. z-score with training stats.
    Xtr, Xtmp, ytr, ytmp = train_test_split(X, y, test_size=(VAL_FRAC + TEST_FRAC),
                                            stratify=y, random_state=seed)
    rel = TEST_FRAC / (VAL_FRAC + TEST_FRAC)
    Xva, Xte, yva, yte = train_test_split(Xtmp, ytmp, test_size=rel,
                                          stratify=ytmp, random_state=seed)
    imp = SimpleImputer(strategy='median').fit(Xtr)
    Xtr, Xva, Xte = imp.transform(Xtr), imp.transform(Xva), imp.transform(Xte)
    sc = StandardScaler().fit(Xtr)
    return sc.transform(Xtr), ytr, sc.transform(Xva), yva, sc.transform(Xte), yte

def score_final(builder, params, Xtr, ytr, Xte, yte, seed):
    clf = builder(params, seed)
    clf.fit(Xtr, ytr)
    yp = clf.predict(Xte)
    acc = accuracy_score(yte, yp)
    f1_pos = f1_score(yte, yp, average='binary', pos_label=1, zero_division=0)
    f1_macro = f1_score(yte, yp, average='macro', zero_division=0)
    tn, fp, fn, tp = confusion_matrix(yte, yp, labels=[0, 1]).ravel()
    fnr = fn / max(1, (fn + tp))
    return acc, f1_pos, f1_macro, fnr

# ---------------------------------------------------------------------------
# INCREMENTAL SAVES -- disconnect-resilient.
# After EVERY (dataset, classifier, method) finishes all 10 seeds, the full
# records list is rewritten to /content/results_raw.csv AND appended to a
# /content/results_partial/ checkpoint directory. If the runtime disconnects
# partway through, rerun this cell: the outer loops skip any
# (dataset, classifier, method) block already present in the checkpoint.
# ---------------------------------------------------------------------------
import os, glob
CKPT_DIR = '/content/results_partial'
os.makedirs(CKPT_DIR, exist_ok=True)

def ckpt_path(ds, clf, method):
    return f'{CKPT_DIR}/{ds}__{clf}__{method}.csv'

def load_all_checkpoints():
    files = sorted(glob.glob(f'{CKPT_DIR}/*.csv'))
    if not files:
        return []
    dfs = [pd.read_csv(f) for f in files]
    return pd.concat(dfs, ignore_index=True).to_dict('records')

# Bootstrap: if a previous successful run left /content/results_raw.csv
# but no checkpoint files, treat that full CSV as equivalent to having
# every block checkpointed. Rerunning this cell then becomes a no-op.
if (not glob.glob(f'{CKPT_DIR}/*.csv')) and os.path.exists('/content/results_raw.csv'):
    print('Found /content/results_raw.csv from a previous run -- seeding checkpoints.')
    prev = pd.read_csv('/content/results_raw.csv')
    for (ds, clf, m), g in prev.groupby(['dataset', 'classifier', 'method']):
        g.to_csv(ckpt_path(ds, clf, m), index=False)

records = load_all_checkpoints()
done_keys = {(r['dataset'], r['classifier'], r['method']) for r in records}
if done_keys:
    print(f'Resuming -- {len(done_keys)} (dataset, clf, method) blocks already done.')

t_start = time.perf_counter()
for ds_name in ACTIVE_DATASETS:
    X, y = all_datasets[ds_name]
    for clf_name, (builder, space) in CLASSIFIERS.items():
        for method_name, method_fn in HPO_METHODS.items():
            if (ds_name, clf_name, method_name) in done_keys:
                elapsed = time.perf_counter() - t_start
                print(f'[{elapsed/60:5.1f} min] {ds_name:<5} {clf_name:<4} '
                      f'{method_name:<8} SKIP (checkpoint).')
                continue
            block = []
            for seed in SEEDS:
                np.random.seed(seed); random.seed(seed)
                Xtr, ytr, Xva, yva, Xte, yte = prep_split(X, y, seed)
                best_params, n_evals, t_opt = method_fn(clf_name, builder, space,
                                                        Xtr, ytr, Xva, yva, seed)
                acc, f1p, f1m, fnr = score_final(builder, best_params,
                                                 Xtr, ytr, Xte, yte, seed)
                row = {
                    'dataset': ds_name, 'classifier': clf_name, 'method': method_name,
                    'seed': seed, 'accuracy': acc, 'f1_pos': f1p, 'f1_macro': f1m,
                    'fnr': fnr, 'n_evals': n_evals, 'time_s': t_opt,
                    'best_params': json.dumps(best_params, default=str),
                }
                block.append(row)
                records.append(row)
            # Write block-level checkpoint + rolling full CSV
            pd.DataFrame(block).to_csv(ckpt_path(ds_name, clf_name, method_name), index=False)
            pd.DataFrame(records).to_csv('/content/results_raw.csv', index=False)
            elapsed = time.perf_counter() - t_start
            print(f'[{elapsed/60:5.1f} min] {ds_name:<5} {clf_name:<4} '
                  f'{method_name:<8} done (10 seeds, {len(records)} rows cumulative).')

results_raw = pd.DataFrame(records)
results_raw.to_csv('/content/results_raw.csv', index=False)
print(f'\nWrote /content/results_raw.csv ({len(results_raw)} rows)')
print(f'Total wall-clock: {(time.perf_counter()-t_start)/60:.1f} minutes')
print(f'Checkpoints in {CKPT_DIR} -- safe to delete once results_raw.csv is verified.')


## 7. Aggregate, run statistical tests, compute effect sizes

In [ ]:
from scipy.stats import wilcoxon

# If the runtime was restarted between running cell 16 and this cell,
# `results_raw` is no longer in memory -- reload from disk.
try:
    results_raw
except NameError:
    print('results_raw not in memory, loading /content/results_raw.csv ...')
    results_raw = pd.read_csv('/content/results_raw.csv')
print(f'results_raw: {len(results_raw)} rows, '
      f'{results_raw["dataset"].nunique()} datasets, '
      f'{results_raw["method"].nunique()} methods.')

def cliffs_delta(x, y):
    x, y = np.asarray(x), np.asarray(y)
    gt = np.sum(x[:, None] > y[None, :])
    lt = np.sum(x[:, None] < y[None, :])
    return (gt - lt) / (len(x) * len(y))

def holm_bonferroni(p_values):
    n = len(p_values)
    order = np.argsort(p_values)
    adjusted = np.empty(n)
    running_max = 0.0
    for rank, idx in enumerate(order):
        adj = p_values[idx] * (n - rank)
        running_max = max(running_max, adj)
        adjusted[idx] = min(1.0, running_max)
    return adjusted

summary_rows = []
# For each (dataset, classifier), compute mean/std per method and compare to Manual
for (ds, clf), g in results_raw.groupby(['dataset', 'classifier']):
    manual_acc = g[g['method'] == 'Manual']['accuracy'].values
    entries = []
    for m, gm in g.groupby('method'):
        acc = gm['accuracy'].values
        f1m = gm['f1_macro'].values
        fnr = gm['fnr'].values
        t   = gm['time_s'].values
        entry = {
            'dataset': ds, 'classifier': clf, 'method': m,
            'acc_mean': acc.mean(), 'acc_std': acc.std(ddof=1),
            'f1m_mean': f1m.mean(), 'f1m_std': f1m.std(ddof=1),
            'fnr_mean': fnr.mean()*100, 'fnr_std': fnr.std(ddof=1)*100,
            'time_mean': t.mean(), 'time_std': t.std(ddof=1),
        }
        if m == 'Manual':
            entry['p_value'] = np.nan
            entry['cliffs_delta'] = 0.0
        else:
            try:
                if np.all(acc - manual_acc == 0):
                    entry['p_value'] = 1.0
                else:
                    stat, p = wilcoxon(acc, manual_acc, zero_method='wilcox',
                                       alternative='two-sided', mode='auto')
                    entry['p_value'] = float(p)
            except Exception:
                entry['p_value'] = np.nan
            entry['cliffs_delta'] = float(cliffs_delta(acc, manual_acc))
        entries.append(entry)
    # Holm-Bonferroni within this (dataset, classifier) family of comparisons
    non_manual = [e for e in entries if e['method'] != 'Manual']
    pvals = np.array([e['p_value'] if not np.isnan(e['p_value']) else 1.0
                      for e in non_manual])
    adj = holm_bonferroni(pvals)
    for e, a in zip(non_manual, adj):
        e['p_value_holm'] = float(a)
    for e in entries:
        if e['method'] == 'Manual':
            e['p_value_holm'] = np.nan
        summary_rows.append(e)

results_summary = pd.DataFrame(summary_rows)

# Sort by (dataset, classifier, method) with method in a sensible order
method_order = ['Manual', 'Grid', 'Random', 'Bayesian', 'GWO', 'PSO']
results_summary['_method_rank'] = results_summary['method'].map(
    {m: i for i, m in enumerate(method_order)})
results_summary = (results_summary
                   .sort_values(['dataset', 'classifier', '_method_rank'])
                   .drop(columns=['_method_rank'])
                   .reset_index(drop=True))

results_summary.to_csv('/content/results_summary.csv', index=False)
print(f'\nWrote /content/results_summary.csv ({len(results_summary)} rows).')
results_summary.head(20)


## 8. Generate LaTeX rows for the manuscript

Outputs blocks that paste directly into `tab:pima_results`, `tab:ilpd_results`, and `tab:summary` in the .tex file.


In [ ]:
TABLE_LABEL = {'BC': 'tab:bc_results', 'HD': 'tab:hd_results',
               'PD': 'tab:pima_results', 'ILPD': 'tab:ilpd_results'}
METHOD_ORDER = ['Manual', 'Grid', 'Random', 'Bayesian', 'GWO', 'PSO']
CLS_ORDER = ['RF', 'SVM', 'LR']

def fmt_cell(mean, std, digits=4):
    return f'${mean:.{digits}f} \\pm {std:.{digits}f}$'

def fmt_p(p):
    if p is None or (isinstance(p, float) and np.isnan(p)):
        return '--'
    if p < 0.001:
        return '$<0.001$'
    return f'${p:.3f}$'

def generate_rows(df, ds_name):
    sub = df[df['dataset'] == ds_name]
    lines = []
    for clf in CLS_ORDER:
        lines.append(f'\t\t% --- {ds_name} / {clf} ---')
        lines.append(f'\t\t\\multirow{{6}}{{*}}{{{clf}}}')
        for method in METHOD_ORDER:
            r = sub[(sub['classifier'] == clf) & (sub['method'] == method)]
            if len(r) == 0: continue
            r = r.iloc[0]
            acc = fmt_cell(r['acc_mean'], r['acc_std'])
            f1m = fmt_cell(r['f1m_mean'], r['f1m_std'])
            fnr = fmt_cell(r['fnr_mean'], r['fnr_std'], digits=2)
            tim = fmt_cell(r['time_mean'], r['time_std'], digits=2)
            p   = fmt_p(r.get('p_value'))
            lines.append(f'\t\t& {method:<8} & {acc} & {f1m} & {fnr} & {tim} & {p} \\\\')
        lines.append('\t\t\\hline')
    return '\n'.join(lines)

latex_out = []
for ds in ACTIVE_DATASETS:
    latex_out.append(f'% ======== Rows for {TABLE_LABEL[ds]} ========')
    latex_out.append(generate_rows(results_summary, ds))
    latex_out.append('')

# Cross-dataset summary rows
def best_row_for(ds, clf):
    sub = results_summary[(results_summary['dataset']==ds) & (results_summary['classifier']==clf)]
    if len(sub)==0: return None
    i = sub['acc_mean'].idxmax()
    r = sub.loc[i]
    return r['method'], r['acc_mean'], r.get('p_value', np.nan)

latex_out.append('% ======== Rows for tab:summary (cross-dataset) ========')
n_by_ds = {'BC':569,'HD':270,'PD':768,'ILPD':583}
for ds in ACTIVE_DATASETS:
    latex_out.append(f'\t\t\\multirow{{3}}{{*}}{{{ds} (${n_by_ds.get(ds,"?")}$)}}')
    for clf in CLS_ORDER:
        br = best_row_for(ds, clf)
        if br is None: continue
        m, acc, p = br
        latex_out.append(f'\t\t& {clf} & {m} & ${acc:.4f}$ & {fmt_p(p)} \\\\')
    latex_out.append('\t\t\\hline')

latex_text = '\n'.join(latex_out)
with open('/content/latex_tables.tex', 'w') as f:
    f.write(latex_text)

print(latex_text)
print('\n\nWrote /content/latex_tables.tex')

## 9. Download outputs

The three files to grab: `results_raw.csv`, `results_summary.csv`, `latex_tables.tex` (and `dataset_hashes.json` for the Reproducibility section).


In [ ]:
import shutil
if MOUNT_DRIVE:
    for name in ['results_raw.csv', 'results_summary.csv', 'latex_tables.tex']:
        shutil.copy(f'/content/{name}', f'{DRIVE_OUTPUT_DIR}/{name}')
    shutil.copy('/content/data/dataset_hashes.json', f'{DRIVE_OUTPUT_DIR}/dataset_hashes.json')
    print('Copied to Drive:', DRIVE_OUTPUT_DIR)

try:
    from google.colab import files
    for name in ['results_raw.csv', 'results_summary.csv', 'latex_tables.tex']:
        files.download(f'/content/{name}')
except Exception as e:
    print('Download trigger (only runs in Colab):', e)
    print('\nManual download: open the Files panel on the left and right-click to download.')

## 10. After the run — paste into the manuscript

1. Open `latex_tables.tex` (text file).
2. Copy the block under `% ======== Rows for tab:pima_results ========` and paste it into `root_journal.tex` replacing the placeholder `---` rows under `tab:pima_results`.
3. Do the same for `tab:ilpd_results`.
4. Copy the `% ======== Rows for tab:summary ========` block and replace the PD/ILPD placeholder rows in `tab:summary`.
5. Paste `dataset_hashes.json` values into the Reproducibility section.
6. Write the 6 interpretive paragraphs (RF / SVM / LR × PD / ILPD) using the existing BC/HD paragraphs as templates, referencing your new numbers.
7. Rewrite the abstract and the cross-dataset discussion paragraph to reflect the four-dataset picture.

The reproducibility chain is: pinned library versions in cell 2, seeded splits in §6, SHA-256 hashes in §3 for dataset provenance, Holm-Bonferroni + Cliff's δ in §7 as reviewer-defensible statistics.
